# Notebook 06 — Finalize, Save, and Verify the Model




## 1. Imports and project paths


In [1]:

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
NORMALS_PER_FRAUD = 20

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "processed").exists() and (PROJECT_ROOT.parent / "data" / "processed").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "transactions_features.csv.gz"
SPLIT_CONFIG_PATH = PROJECT_ROOT / "notebooks" / "outputs" / "split_config.json"
THRESHOLD_REPORT_PATH = PROJECT_ROOT / "notebooks" / "outputs" / "validation_threshold_results.csv"
OUTPUT_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = OUTPUT_DIR / "fraud_detection_pipeline.pkl"
METADATA_PATH = OUTPUT_DIR / "fraud_detection_metadata.json"

for path in [DATA_PATH, SPLIT_CONFIG_PATH, THRESHOLD_REPORT_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

print("Project root:", PROJECT_ROOT)
print("Model output folder:", OUTPUT_DIR)


Project root: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system
Model output folder: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system\models


## 2. Load data and recreate the chronological training sample


In [2]:

with open(SPLIT_CONFIG_PATH, "r", encoding="utf-8") as f:
    split_config = json.load(f)

TARGET = split_config["target_column"]
TIME_COL = split_config["time_column"]
CUTOFF_STEP = int(split_config["cutoff_step"])

df = pd.read_csv(DATA_PATH, compression="gzip")
outer_train = df.loc[df[TIME_COL] <= CUTOFF_STEP].copy()
test_df = df.loc[df[TIME_COL] > CUTOFF_STEP].copy()

train_steps = np.sort(outer_train[TIME_COL].dropna().unique())
inner_position = int(np.floor(len(train_steps) * 0.80)) - 1
inner_position = max(0, min(inner_position, len(train_steps) - 2))
VALIDATION_CUTOFF = int(train_steps[inner_position])

train_df = outer_train.loc[outer_train[TIME_COL] <= VALIDATION_CUTOFF].copy()
valid_df = outer_train.loc[outer_train[TIME_COL] > VALIDATION_CUTOFF].copy()

excluded = {
    TARGET, "isFlaggedFraud", "nameOrig", "nameDest", TIME_COL,
    "newbalanceOrig", "newbalanceDest",
    "origin_balance_error", "destination_balance_error",
    "origin_balance_error_abs", "destination_balance_error_abs",
    "origin_emptied", "destination_balance_zero"
}
feature_columns = [c for c in df.columns if c not in excluded]

# Guardrails against known post-transaction / target / identity columns.
for forbidden in [
    TARGET, "isFlaggedFraud", "nameOrig", "nameDest", TIME_COL,
    "newbalanceOrig", "newbalanceDest", "origin_balance_error",
    "destination_balance_error", "origin_balance_error_abs",
    "destination_balance_error_abs", "origin_emptied",
    "destination_balance_zero"
]:
    assert forbidden not in feature_columns, f"Forbidden feature found: {forbidden}"

train_fraud = train_df.loc[train_df[TARGET] == 1]
train_normal = train_df.loc[train_df[TARGET] == 0]
if train_fraud.empty:
    raise ValueError("No fraud examples in the training partition.")

normal_sample_size = min(len(train_normal), len(train_fraud) * NORMALS_PER_FRAUD)
train_normal_sample = train_normal.sample(n=normal_sample_size, random_state=RANDOM_STATE)
train_sample = pd.concat([train_fraud, train_normal_sample], ignore_index=False)
train_sample = train_sample.sample(frac=1, random_state=RANDOM_STATE)

X_train = train_sample[feature_columns].copy()
y_train = train_sample[TARGET].astype("int8").copy()

categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = [c for c in feature_columns if c not in categorical_features]
for col in numeric_features:
    X_train[col] = pd.to_numeric(X_train[col], errors="coerce")

print(f"Training rows: {len(X_train):,}")
print(f"Fraud rows: {int(y_train.sum()):,}")
print(f"Normal rows: {int((y_train == 0).sum()):,}")
print(f"Feature count: {len(feature_columns)}")
print("Features:", feature_columns)
print("Chronological train/validation/test separation:", train_df[TIME_COL].max() < valid_df[TIME_COL].min() < test_df[TIME_COL].min())


Training rows: 111,615
Fraud rows: 5,315
Normal rows: 106,300
Feature count: 9
Features: ['type', 'amount', 'oldbalanceOrg', 'oldbalanceDest', 'hour_of_day', 'night_flag', 'amount_to_origin_balance_ratio', 'origin_balance_was_zero', 'origin_txn_count_prior_steps']
Chronological train/validation/test separation: True


C:\Users\Tanish.Ranjan\AppData\Local\Temp\ipykernel_21404\1234055736.py:52: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()


## 3. Read the validation-selected threshold
The threshold comes from Notebook 05 validation results. 

In [3]:

threshold_report = pd.read_csv(THRESHOLD_REPORT_PATH)
MODEL_NAME = "Random Forest (balanced)"
selected = threshold_report.loc[threshold_report["model"] == MODEL_NAME]
if selected.empty:
    raise ValueError(
        f"{MODEL_NAME!r} not found in {THRESHOLD_REPORT_PATH}. "
        "Open the report and confirm the model label."
    )

THRESHOLD = float(selected.iloc[0]["threshold"])
print(f"Model: {MODEL_NAME}")
print(f"Validation-selected threshold: {THRESHOLD:.6f}")
print("Validation F2:", round(float(selected.iloc[0]["f2"]), 6))
print("Validation precision:", round(float(selected.iloc[0]["precision"]), 6))
print("Validation recall:", round(float(selected.iloc[0]["recall"]), 6))


Model: Random Forest (balanced)
Validation-selected threshold: 0.800000
Validation F2: 0.890688
Validation precision: 0.917431
Validation recall: 0.884244


## 4. Build and train the full preprocessing + model pipeline
The preprocessing steps are inside the pipeline so future predictions use the same transformations.

In [4]:

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features)
    ],
    remainder="drop"
)

final_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=14,
        min_samples_leaf=4,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

final_pipeline.fit(X_train, y_train)
print("Pipeline training complete.")


Pipeline training complete.


## 5. Save the pipeline and metadata

In [5]:

artifact = {
    "pipeline": final_pipeline,
    "threshold": THRESHOLD,
    "feature_columns": feature_columns,
    "target_column": TARGET,
    "time_column": TIME_COL,
    "training_cutoff_step": VALIDATION_CUTOFF,
    "outer_training_cutoff_step": CUTOFF_STEP,
    "model_name": MODEL_NAME
}
joblib.dump(artifact, MODEL_PATH)

metadata = {
    "model_name": MODEL_NAME,
    "threshold": THRESHOLD,
    "feature_columns": feature_columns,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "target_column": TARGET,
    "time_column": TIME_COL,
    "training_sample_rows": int(len(X_train)),
    "training_fraud_rows": int(y_train.sum()),
    "training_normal_rows": int((y_train == 0).sum()),
    "random_state": RANDOM_STATE,
    "normals_per_fraud_cap": NORMALS_PER_FRAUD,
    "training_cutoff_step": VALIDATION_CUTOFF,
    "outer_training_cutoff_step": CUTOFF_STEP,
    "warning": "Reproducibility checks do not prove absence of leakage or production readiness."
}
with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved pipeline:", MODEL_PATH)
print("Saved metadata:", METADATA_PATH)
print("Pipeline file size (MB):", round(MODEL_PATH.stat().st_size / (1024 * 1024), 2))


Saved pipeline: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system\models\fraud_detection_pipeline.pkl
Saved metadata: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system\models\fraud_detection_metadata.json
Pipeline file size (MB): 2.87


## 6. Verify predictions are identical after reloading
checks both fraud probabilities and labels from the saved artifact. A mismatch should stop Phase 1 completion.

In [9]:

# Verify predictions before and after reloading the saved pipeline

reloaded_artifact = joblib.load(MODEL_PATH)

reloaded_pipeline = reloaded_artifact["pipeline"]
reloaded_threshold = float(reloaded_artifact["threshold"])
reloaded_features = reloaded_artifact["feature_columns"]

# Use the same test sample for both predictions
test_sample = test_df.sample(
    n=min(1000, len(test_df)),
    random_state=RANDOM_STATE
)

X_check = test_sample[reloaded_features].copy()

for col in numeric_features:
    X_check[col] = pd.to_numeric(
        X_check[col], errors="coerce"
    )

# Generate probabilities with the original and reloaded pipelines
prob_before = final_pipeline.predict_proba(X_check)[:, 1]
prob_after = reloaded_pipeline.predict_proba(X_check)[:, 1]

# Convert probabilities into fraud/normal labels
pred_before = (prob_before >= THRESHOLD).astype("int8")
pred_after = (prob_after >= reloaded_threshold).astype("int8")

# Allow tiny floating-point differences
probabilities_close = np.allclose(
    prob_before,
    prob_after,
    rtol=1e-12,
    atol=1e-12
)

labels_identical = np.array_equal(
    pred_before,
    pred_after
)

max_difference = float(
    np.max(np.abs(prob_before - prob_after))
)

print("Probability predictions match within tolerance:", probabilities_close)
print("Threshold-based labels identical:", labels_identical)
print("Maximum absolute probability difference:", max_difference)

assert probabilities_close, (
    "Reloaded pipeline probabilities differ beyond the allowed tolerance."
)

assert labels_identical, (
    "Reloaded pipeline labels differ from the original pipeline."
)

print("\nPASS: Saved pipeline reproduces equivalent predictions.")


Probability predictions match within tolerance: True
Threshold-based labels identical: True
Maximum absolute probability difference: 2.220446049250313e-16

PASS: Saved pipeline reproduces equivalent predictions.


## 7. Final Phase 1 artifact summary

In [11]:

# PHASE 1 — FINAL MODEL ARTIFACT SUMMARY

print("=" * 60)
print("PHASE 1 — MODEL ARTIFACT SUMMARY")
print("=" * 60)

# 1. Check saved artifacts
pipeline_exists = MODEL_PATH.exists()
metadata_exists = METADATA_PATH.exists()

print("\n1. SAVED ARTIFACTS")
print("-" * 40)
print("Pipeline exists:", pipeline_exists)
print("Metadata exists:", metadata_exists)

# 2. Model configuration
print("\n2. MODEL CONFIGURATION")
print("-" * 40)
print("Model name:", reloaded_artifact["model_name"])
print("Features included:", len(reloaded_features))
print("Classification threshold:", reloaded_threshold)
print("Target column:", reloaded_artifact["target_column"])
print("Time column:", reloaded_artifact["time_column"])

print("\nFeatures used:")
for i, feature in enumerate(reloaded_features, start=1):
    print(f"{i}. {feature}")

# 3. Training information
print("\n3. TRAINING INFORMATION")
print("-" * 40)
print("Training sample rows:", len(X_train))
print("Training fraud rows:", int(y_train.sum()))
print("Training normal rows:", int((y_train == 0).sum()))

# 4. Reload verification
# These variables must be created by the updated verification cell:
# probabilities_close, labels_identical, max_difference

print("\n4. SAVED MODEL VERIFICATION")
print("-" * 40)

if "probabilities_close" in globals():
    print(
        "Probability predictions match within tolerance:",
        probabilities_close
    )
else:
    probabilities_close = False
    print("Probability verification: NOT RUN")

if "labels_identical" in globals():
    print("Threshold-based labels identical:", labels_identical)
else:
    labels_identical = False
    print("Label verification: NOT RUN")

if "max_difference" in globals():
    print("Maximum probability difference:", max_difference)
else:
    print("Maximum probability difference: NOT AVAILABLE")

verification_passed = (
    probabilities_close and labels_identical
)

print(
    "Reloaded predictions equivalent:",
    verification_passed
)

# 5. Final status
print("\n5. PHASE 1 STATUS")
print("-" * 40)

artifacts_saved = pipeline_exists and metadata_exists

if artifacts_saved and verification_passed:
    print("PASS: Model artifacts saved and prediction verification passed.")
else:
    print("INCOMPLETE: Review the artifact or verification checks above.")

print("\nREMAINING REVIEW BEFORE FINAL PHASE 1 SIGN-OFF")
print("-" * 40)
print("1. Audit all features for target leakage and post-transaction data.")
print("2. Confirm the threshold was selected using validation data only.")
print("3. Record final test metrics and the confusion matrix from Notebook 05.")
print("4. Document limitations and investigate unusually high model scores.")
print("5. Complete any required time-aware cross-validation checks.")

print("\nNote: Artifact verification does not prove production readiness.")
print("=" * 60)


PHASE 1 — MODEL ARTIFACT SUMMARY

1. SAVED ARTIFACTS
----------------------------------------
Pipeline exists: True
Metadata exists: True

2. MODEL CONFIGURATION
----------------------------------------
Model name: Random Forest (balanced)
Features included: 9
Classification threshold: 0.7999999999999999
Target column: isFraud
Time column: step

Features used:
1. type
2. amount
3. oldbalanceOrg
4. oldbalanceDest
5. hour_of_day
6. night_flag
7. amount_to_origin_balance_ratio
8. origin_balance_was_zero
9. origin_txn_count_prior_steps

3. TRAINING INFORMATION
----------------------------------------
Training sample rows: 111615
Training fraud rows: 5315
Training normal rows: 106300

4. SAVED MODEL VERIFICATION
----------------------------------------
Probability predictions match within tolerance: True
Threshold-based labels identical: True
Maximum probability difference: 2.220446049250313e-16
Reloaded predictions equivalent: True

5. PHASE 1 STATUS
---------------------------------------